<a href="https://colab.research.google.com/github/allenbasicdev/The-Crossroads/blob/main/The_Crossroads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install uv
!uv pip install guidance
!uv pip install transformers
!uv pip install huggingface_hub
!uv pip install outlines
!uv pip install pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 28.8 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 35 packages in 1.21s
Prepared 7 packages in 2.09s
Uninstalled 2 packages in 16ms
Installed 7 packages in 445ms
 + comm==0.2.3
 + guidance==0.3.1
 + guidance-stitch==0.1.5
 - ipywidgets==7.7.1
 + ipywidgets==8.1.9
 + jedi==0.20.0
 + llguidance==1.5.0
 - widgetsnbextension==3.6.10
 + widgetsnbextension==4.0.16
Using Python 3.13.15 environment at: /usr
Checked 1 package in 151ms
Using Python 3.13.15 environment at: /usr
Checked 1 package in 246ms
Using Python 3.13.15 environment at: /usr
Resolved 18 packages in 573ms
Prepared 4 packages in 244ms
Installed 4 packages in 20ms
 + diskcache==5.6.3
 + genson==1.4.0
 + outlines==1.3.3
 + outlines-core==0.2.14
Using Python 3.13.15 environment at: /usr
Checked 1 package in 351ms


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id = "Qwen/Qwen2.5-0.5B-Instruct", local_dir = "model")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'/content/model'

In [ ]:
import torch
import outlines
from outlines import models
from transformers import AutoModelForCausalLM, AutoTokenizer
from guidance import models, gen, json

model = AutoModelForCausalLM.from_pretrained("model", device_map = "cuda", trust_remote_code = True)
tokenize = AutoTokenizer.from_pretrained("model", trust_remote_code = True)

modelreal = models.Transformers(model, tokenize)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, create_model
from enum import Enum
from guidance import select
import numpy as np

def findbest(optionsnames, options, obj, desc, action):
    prompt = "Pay extreme attention to actions and negative words. \nThere is a " + obj + ", " + desc + ".\n You" + action + "\n"

    prompttokens = (tokenize(prompt, return_tensors="pt").to("cuda"))["input_ids"].shape[1]

    tokenize.padding_side = "right"
    if tokenize.pad_token is None:
        tokenize.pad_token = tokenize.eos_token

    answersfull = []
    for i in range(0, len(options)):
        answersfull.append(prompt + options[i])

    inputs = tokenize(answersfull, return_tensors="pt", padding = True).to("cuda")
    inputids = inputs["input_ids"]
    attentionmask = inputs["attention_mask"]

    with torch.no_grad():
        outputs = model(**inputs)
        outputlogits = outputs.logits

    logitshift = outputlogits[..., :-1, :].contiguous()
    labelshift = inputids[..., 1:].contiguous()
    attentionmaskshift = attentionmask[..., 1:].contiguous()

    logprobs = torch.nn.functional.log_softmax(logitshift, dim=-1)
    alllogprobs = torch.gather(logprobs, dim=-1, index = labelshift.unsqueeze(-1)).squeeze(-1) * attentionmaskshift

    scores = []
    for i in range(0, len(options)):
        thislogprobs = alllogprobs[i]
        logprobsreal = thislogprobs[attentionmaskshift[i].bool()][(prompttokens - 1):]

        scores.append(logprobsreal.mean().item())
        print(logprobsreal.mean().item())

    maxindex = 0; maxval = scores[0]
    for i in range(0, len(options)):
        if scores[i] > maxval:
            maxindex = i
            maxval = scores[i]

    print(optionsnames[maxindex])


In [ ]:
obj = "tree"
desc = "just a normal tree you would find in your backyard"
action = "cut down"
optionsdesc = ["the tree is on fire", "the tree is burnt", "there is only a stump left of the tree", "the tree vanished into thin air", "the tree is uprooted", "the tree grew", "nothing much happens"]
options = ["TreeOnFire", "TreeBurnt", "TreeStump", "TreeVanish", "TreeUprooted", "TreeGrow", "Nothing"]

findbest(options, optionsdesc, obj, desc, action)


-3.5625
-5.1875
-2.609375
-4.15625
-3.59375
-5.75
-6.0625
TreeStump


In [ ]:
obj = "lamp"
desc = "a bedside lamp"
action = "REMOVE the filament"
optionsdesc = ["the lamp is now on", "the lamp is now off", "the lamp is now on fire", "the lamp is now broken", "the lamp disappeared", "nothing much happens"]
options = ["LampOn", "LampOff", "LampOnFire", "LampBroken", "LampDisappear", "Nothing"]

findbest(options, optionsdesc, obj, desc, action)


-3.109375
-3.09375
-4.0625
-3.09375
-6.0625
-6.125
LampOff


In [ ]:
obj = "person"
desc = "a person who just baked a batch of cookies that they do NOT want you to eat"
action = "eat the cookies"
optionsdesc = ["the person is happy", "the person is sad", "the person is angry", "the person is dead", "the person is scared", "nothing much happens"]
options = ["Happy", "Sad", "Angry", "Dead", "Scared", "Nothing"]

findbest(options, optionsdesc, obj, desc, action)

-4.03125
-4.15625
-3.859375
-4.875
-4.71875
-6.3125
Angry
